# Spotify Recommender System – Part 4: ALS Collaborative Filtering (Optuna Tuning)
**Author:** Miguel Vásquez  
**Date:** 23 September 2025  
**Updated:** 03 October 2025 

---

## 1. Introduction  

This notebook trains a **Collaborative Filtering model** using the **Alternating Least Squares (ALS)** algorithm from the implicit library, **with hyperparameter tuning via Optuna**.  
Previous attempts with LightFM were not feasible due to memory constraints, so **ALS is the primary model**.

ALS is an efficient matrix factorization method for **large, sparse datasets**. It was chosen as the primary model in this project because it scales better than LightFM and allows users to exploit implicit interactions with songs.

Workflow:

- **03_data_prep.ipynb** → preprocess data and build interaction matrices.
- **04_ALS_training.ipynb** → train ALS with Optuna tuning, save the best model and study logs.
- **02_baseline_popularity.ipynb** → establish and evaluate the popularity baseline.
- **Evaluation and comparison** → integrated within the baseline and ALS notebooks, instead of a separate file.

---

## 2. Objectives of this notebook  

- Load the preprocessed interaction matrices from `data/processed/`.  
- Train an ALS model with tuned hyperparameters.  
- Persist the trained model to `models/als_model.pkl`.  
- Verify model training with basic logs.  

The comprehensive **evaluation** (metrics, baseline comparison, and conclusions) is **performed in Notebook 05**, to keep **this notebook focused solely on training**.

In [1]:
import os
from pathlib import Path
import pickle
import optuna
from IPython.display import display, Markdown
import os

# For OpenBLAS, MKL, NumExpr, OpenMP
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# Add parent directory for imports
import sys
sys.path.append(str(Path.cwd().parent))

# Our ALS training function
from src.load_data import load_data
from src.evaluate import evaluate
from src.models import train_als

# Ensure working dir
if os.getcwd().endswith("notebooks"):
    os.chdir("..")
    display(Markdown(f"Changed working dir to {os.getcwd()}"))

with open("data/processed/item2idx.pkl", "rb") as f:
    item2idx = pickle.load(f)

# Create the inverse
idx2item = {v: k for k, v in item2idx.items()}

# Paths
processed_dir = Path("data/processed")
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

als_path = models_dir / "als_optuna.pkl"
als_params_path = models_dir / "als_optuna_params.pkl"
als_study_path = models_dir / "als_optuna_study.csv"
als_eval_path = models_dir / "als_evaluation.csv"
comparison_eval_path = models_dir / "comparison_evaluation.csv"

c:\Users\Miguel\portfolio\Spotify-Recommender-System\venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Changed working dir to c:\Users\Miguel\portfolio\Spotify-Recommender-System

## 3. Load Data

In [2]:
train_csr, test_csr, interactions, wmat = load_data(processed_dir, weighting="tfidf")
print(f"train: {train_csr.shape}, test: {test_csr.shape}, wmat: {wmat.shape}")

train: (500000, 1614034), test: (500000, 1614034), wmat: (1614034, 500000)


## 4. Optuna
### 4.1 Objective for Optuna

In [3]:
import multiprocessing

def make_objective(wmat, train_csr, test_csr, user_limit=1000, n_jobs=1):
    def objective(trial):
        factors = trial.suggest_int("factors", 32, 256)
        reg = trial.suggest_float("regularization", 1e-3, 1e-1, log=True)
        iters = trial.suggest_int("iterations", 10, 100)

        # Dynamic adjustment according to n_jobs
        n_threads = max(1, multiprocessing.cpu_count() // n_jobs)

        model = train_als(
            wmat, 
            factors=factors, 
            regularization=reg, 
            iterations=iters, 
            num_threads=n_threads
        )

        metrics = evaluate(model, train_csr, test_csr, user_limit=user_limit)
        return metrics["ndcg@100"]
    return objective

### 4.2 Train Optuna

In [ ]:
import pandas as pd

if als_params_path.exists() and als_study_path.exists():
    # Load best params
    with open(als_params_path, "rb") as f:
        best_params = pickle.load(f)
    study_df = pd.read_csv(als_study_path)

    display(Markdown(f"Optuna results already exist. Loaded from:"))
    display(Markdown(f"- Best hyperparameters from `{als_params_path}`"))
    display(Markdown(f"- Study results from `{als_study_path}`"))
    display(Markdown(f"Best params: {best_params}"))
else:
    # Create objective function
    objective = make_objective(wmat, train_csr, test_csr, user_limit=1000)

    # Create study
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    # Run search (example: 50 trials)
    study.optimize(objective, n_trials=50)

    # Save best result
    best_params = study.best_trial.params
    with open(als_params_path, "wb") as f:
        pickle.dump(best_params, f)

    study_df = study.trials_dataframe()
    study_df.to_csv(als_study_path, index=False)

    display(Markdown(f"✅ Best hyperparameters saved to `{als_params_path}`"))
    display(Markdown(f"✅ Study results saved to `{als_study_path}`"))
    display(Markdown(f"Best params: {best_params}"))
    display(Markdown(f"Best metrics: {getattr(study.best_trial, 'user_attrs', {})}"))

Optuna results already exist. Loaded from:

- Best hyperparameters from `models\als_optuna_params.pkl`

- Study results from `models\als_optuna_study.csv`

Best params: {'factors': 243, 'regularization': 0.002320871648621119, 'iterations': 58}

### 4.3 Retrain best model

In [5]:
if als_path.exists():
    with open(als_path, "rb") as f:
        best_model = pickle.load(f)
    display(Markdown(f"ALS model already exists. Loaded from `{als_path}`"))
else:
    best_model = train_als(
        wmat=wmat,
        factors=best_params["factors"],
        regularization=best_params["regularization"],
        iterations=best_params["iterations"],
    )

    with open(als_path, "wb") as f:
        pickle.dump(best_model, f)

    display(Markdown(f"✅ Best ALS model saved to `{als_path}`"))

ALS model already exists. Loaded from `models\als_optuna.pkl`

## 5. Evaluation
In this section we load the tuned ALS model, generate top-K recommendations, compute ranking metrics, and compare results against the Popularity baseline.
- Load ALS model from als_optuna.pkl
- Generate recommendations on test split
- Compute metrics (Precision@K, Recall@K, HitRate@K, NDCG@K)
- Compare against Popularity baseline results
- Save summary table

### 5.1 Evaluate model

In [6]:
if als_eval_path.exists():
    # Load existing results
    als_eval_df = pd.read_csv(als_eval_path)
    display(Markdown("### ALS Model - Evaluation Metrics (loaded from file)"))
    display(als_eval_df)
else:
    Ks = [10, 50, 100]
    results = []
    for k in Ks:
        eval_metrics = evaluate(best_model, train_csr, test_csr, user_limit=5000)
        results.append(eval_metrics)

    # Create DataFrame for evaluation results
    als_eval_df = pd.DataFrame(results)
    display(als_eval_df)

    # Save evaluation results
    als_eval_df.to_csv(als_eval_path, index=False)
    display(Markdown(f"✅ ALS evaluation metrics saved to `{als_eval_path}`"))

### ALS Model - Evaluation Metrics (loaded from file)

,precision@10,recall@10,ndcg@10,hitrate@10,precision@50,recall@50,ndcg@50,hitrate@50,precision@100,recall@100,ndcg@100,hitrate@100
0,0.132091,0.120280,0.173703,0.592348,0.064327,0.270303,0.207157,0.779647,0.043838,0.356939,0.239049,0.833534
1,0.132232,0.118185,0.170733,0.589143,0.065693,0.274214,0.206077,0.790865,0.044247,0.358339,0.237221,0.841947
2,0.131719,0.123838,0.174566,0.595757,0.064098,0.276374,0.211016,0.788873,0.043164,0.362826,0.242362,0.841705


### 5.2 Model Comparison

In [7]:
if comparison_eval_path.exists():
    comparison_df = pd.read_csv(comparison_eval_path)
    display(Markdown("### Comparison: ALS vs Popularity baseline (loaded from file)"))
    display(comparison_df)
else:
    from src.metrics import normalize_eval_df

    # Load evaluation results
    pop_eval_path = models_dir / "popularity_model.csv"

    pop_eval_df = normalize_eval_df(pd.read_csv(pop_eval_path), "Popularity")
    als_eval_df = normalize_eval_df(pd.read_csv(als_eval_path), "ALS")

    # Add model column for clarity
    als_eval_df["Model"] = "ALS"
    pop_eval_df["Model"] = "Popularity"

    # Combine results
    comparison_df = pd.concat([pop_eval_df, als_eval_df], ignore_index=True)

    # Sort for readability
    comparison_df = comparison_df.sort_values(by=["Model"]).reset_index(drop=True)

    display(Markdown("### Comparison: ALS vs Popularity baseline"))
    display(comparison_df)

    # Save combined results
    comparison_df.to_csv(comparison_eval_path, index=False)
    display(Markdown(f"✅ Combined evaluation saved to {comparison_eval_path}"))

### Comparison: ALS vs Popularity baseline (loaded from file)

,precision@10,recall@10,hitrate@10,ndcg@10,precision@50,recall@50,hitrate@50,ndcg@50,precision@100,recall@100,hitrate@100,ndcg@100,Model
0,0.131719,0.123838,0.595757,0.174566,0.064098,0.276374,0.788873,0.211016,0.043164,0.362826,0.841705,0.242362,ALS
1,0.007562,0.006723,0.061576,0.008920,0.005970,0.025593,0.183202,0.015308,0.005172,0.044038,0.264227,0.021723,Popularity


### 5.3 Discussion
The evaluation results clearly demonstrate that the **ALS model consistently outperforms the Popularity baseline** across all metrics and cutoff values. At K=10, ALS achieves a precision of ~0.13 compared to only ~0.007 for the baseline, highlighting the significant improvement in recommendation relevance when leveraging collaborative filtering.

In terms of **recall and hit rate**, ALS is able to retrieve a substantially larger fraction of relevant items. For example, at _K=100_, ALS reaches a recall of ~0.36 and a hit rate of ~0.84, whereas the popularity model achieves only ~0.04 and ~0.26 respectively. These differences confirm that ALS not only identifies more relevant items but also ensures users are exposed to them with higher probability.

The **NDCG metric**, which accounts for ranking quality, further validates the advantage of ALS. Higher NDCG scores across all cutoff values indicate that ALS places relevant items closer to the top of the recommendation lists, improving the user experience compared to the baseline.

Overall, the discussion confirms that **personalized collaborative filtering significantly outperforms naive popularity-driven recommendations**, especially for smaller cutoff values (e.g., K=10), which are more representative of real-world recommendation scenarios where users typically only see a handful of top suggestions.

---

## 6. Conclusion & Future Work
In this project, we implemented and evaluated a recommendation system using **Alternating Least Squares (ALS)** and compared it against a simple **Popularity baseline**. The results demonstrate that ALS substantially improves recommendation quality across multiple evaluation metrics (Precision, Recall, HitRate, and NDCG) and cutoff values (K=10, 50, 100). These findings confirm the effectiveness of personalized collaborative filtering methods for large-scale recommendation tasks.

However, some limitations remain. The evaluation was conducted in an **offline setting**, relying solely on historical user–item interactions, which may not fully capture user satisfaction in real-world scenarios. Furthermore, cold-start issues for new users and items remain a challenge, as ALS relies heavily on interaction history.

As **future work**, several directions can be pursued:
- Incorporating **side information** (e.g., user demographics, item metadata) to mitigate cold-start problems and enhance personalization.
- Exploring **sequence-aware or neural models** (e.g., RNNs, Transformers, or graph-based recommenders) to capture temporal and contextual dynamics.
- Performing **online evaluation** through A/B testing to measure the actual business impact and user engagement of the recommendations.
- Investigating **hybrid approaches** that combine collaborative and content-based signals for more robust performance.

By addressing these aspects, future iterations of the system can become more scalable, adaptive, and user-centric, ultimately improving recommendation relevance and user satisfaction in production environments.